# Complete Data Pipeline

This notebook merges the webcam image feature workflow with the PM2.5, ERA5, ARPA and final merge workflow.


## Cell 1 — Set project paths
The notebook is inside `notebook/`; the project root is one level above it.


In [49]:
import sys
from pathlib import Path
# Detect environment
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
# =========================
# Define project root and image directory
# =========================
if IN_COLAB:
    drive.mount("/content/drive")
    # Google Drive image folder
    IMAGE_DIR = Path("/content/drive/MyDrive/webcam_images")
else:
    # notebook/ -> project root
    PROJECT_ROOT = Path.cwd().parent

    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))

    IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "images"
# =========================
# Define data directories
# =========================
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "config"

RAW_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
# =========================
# Check collected images
# =========================
image_files = (
    list(IMAGE_DIR.glob("*.jpg"))
    + list(IMAGE_DIR.glob("*.jpeg"))
    + list(IMAGE_DIR.glob("*.png"))
)
print("Running in Colab:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Image directory:", IMAGE_DIR)
print("Number of images:", len(image_files))

if len(image_files) > 0:
    print("First image:", image_files[0].name)
else:
    print("No images found in image directory.")

Running in Colab: False
Project root: D:\eo\GeoProject_oo
Image directory: D:\eo\GeoProject_oo\data\raw\images
Number of images: 220
First image: 20260301-0400.jpg


## Cell 2 — Import pipeline functions


In [50]:
import importlib

import src.roi_viewer as roi_viewer
import src.image_features as image_features
import src.eea_pm25_download as eea_pm25_download
import src.era5_download as era5_download
import src.merge_arpa_data as merge_arpa_data
import src.merge_all_data as merge_all_data

# Reload modules during development, useful in PyCharm/Jupyter after editing .py files
importlib.reload(roi_viewer)
importlib.reload(image_features)
importlib.reload(eea_pm25_download)
importlib.reload(era5_download)
importlib.reload(merge_arpa_data)
importlib.reload(merge_all_data)

from src.roi_viewer import build_roi_viewer
from src.image_features import extract_image_features
from src.eea_pm25_download import download_pm25_data
from src.era5_download import download_era5_data
from src.merge_arpa_data import merge_arpa_tables
from src.merge_all_data import merge_all_datasets


## Cell 3 — Configure parameters
Only this cell needs to be edited when changing dates, input files, output files, station ID or image folder.


In [61]:
# =========================
# Date range
# =========================
START_DATE = "2026-03-01"
END_DATE = "2026-03-12"

API_START = f"{START_DATE}T00:00:00Z"
API_END = f"{END_DATE}T23:59:59Z"

# =========================
# Location settings
# =========================
LAT = 45.46
LON = 9.19

# PM2.5 monitoring station
STATION_PREFIX = "IT/SPO.IT0477A_6001_BETA"

# ERA5 pressure-level area: North, West, South, East
PRESSURE_AREA = [45.50, 9.00, 45.25, 9.25]

# =========================
# Input folders and files
# =========================
IMAGE_DIR = RAW_DIR / "images"

ARPA_FILES = [
    RAW_DIR / "arpa" /"2026.01.01-2026.05.16"/ "RW_20260517060806_760968_5911_1.csv",
    RAW_DIR / "arpa" /"2026.01.01-2026.05.16"/ "RW_20260517060808_760968_6048_1.csv",
    RAW_DIR / "arpa" /"2026.01.01-2026.05.16"/ "RW_20260517060809_760968_6597_1.csv",
    RAW_DIR / "arpa" /"2026.01.01-2026.05.16"/ "RW_20260517060811_760968_19019_1.csv",
    RAW_DIR / "arpa" /"2026.01.01-2026.05.16"/ "RW_20260517060811_760968_19019_3.csv",
]

# =========================
# Output files
# =========================
ROI_JSON = CONFIG_DIR / "roi.json"
IMAGE_FEATURE_FILE = INTERIM_DIR / "image_features.csv"
PM25_OUTPUT_FILE = INTERIM_DIR / "PM25_MI_hourly.csv"
ERA5_OUTPUT_FILE = INTERIM_DIR / "era5_all_merged.csv"
ARPA_OUTPUT_FILE = INTERIM_DIR / "arpa_merged.csv"
FINAL_OUTPUT_FILE = PROCESSED_DIR / "final_merged_all.csv"

# Temporary folders
PM25_TEMP_DIR = INTERIM_DIR / "pm25_temp"
ERA5_WORK_DIR = RAW_DIR / "era5"

print("Image directory:", IMAGE_DIR)
print("Image features output:", IMAGE_FEATURE_FILE)
print("Final output:", FINAL_OUTPUT_FILE)


Image directory: D:\eo\GeoProject_oo\data\raw\images
Image features output: D:\eo\GeoProject_oo\data\interim\image_features.csv
Final output: D:\eo\GeoProject_oo\data\processed\final_merged_all.csv


## Cell 4 — Load webcam images
This checks whether the image folder exists and counts available images.


In [52]:
image_files = sorted(
    list(IMAGE_DIR.glob("*.jpg"))
    + list(IMAGE_DIR.glob("*.jpeg"))
    + list(IMAGE_DIR.glob("*.png"))
)

print("Image directory:", IMAGE_DIR)
print("Number of images:", len(image_files))

if len(image_files) > 0:
    print("First image:", image_files[0].name)
else:
    print("No images found. Please put webcam images in data/raw/images.")


Image directory: D:\eo\GeoProject_oo\data\raw\images
Number of images: 220
First image: 20260301-0400.jpg


## Cell 5 — Select ROI
Run this cell, adjust the sliders, then run the next cell to save the selected ROI.


In [7]:
roi_controls = build_roi_viewer(IMAGE_DIR)


Output()

## Cell 6 — Save selected ROI


In [27]:
import json

selected_roi = {
    "top": roi_controls["top"].value,
    "left": roi_controls["left"].value,
    "height": roi_controls["height"].value,
    "width": roi_controls["width"].value,
}

ROI_JSON.parent.mkdir(parents=True, exist_ok=True)

with open(ROI_JSON, "w", encoding="utf-8") as file:
    json.dump(selected_roi, file, indent=4)

print("Selected ROI:")
print(f"top = {selected_roi['top']}")
print(f"left = {selected_roi['left']}")
print(f"height = {selected_roi['height']}")
print(f"width = {selected_roi['width']}")
print(f"mode = {roi_controls['mode'].value}")

print("\nROI saved to:")
print(ROI_JSON)


Selected ROI:
top = 160
left = 116
height = 380
width = 515
mode = RGB

ROI saved to:
D:\eo\GeoProject_oo\config\roi.json


## Cell 7 — Extract image features
This creates `data/interim/image_features.csv`. The image filename time is interpreted as Europe/Rome local time and converted to UTC.


In [28]:
import pandas as pd

with open(ROI_JSON, "r", encoding="utf-8") as file:
    selected_roi = json.load(file)

rows, skipped_files = extract_image_features(
    image_dir=IMAGE_DIR,
    output_csv=IMAGE_FEATURE_FILE,
    roi=selected_roi,
    project_root=PROJECT_ROOT,
    image_timezone="Europe/Rome",
    output_timezone="UTC",
)

image_df = pd.read_csv(IMAGE_FEATURE_FILE)

print(f"Done. Wrote {len(rows)} rows.")
print(f"Skipped files: {len(skipped_files)}")

image_df.head()


Saved image features: D:\eo\GeoProject_oo\data\interim\image_features.csv
Rows written: 220
Skipped files: 0
Done. Wrote 220 rows.
Skipped files: 0


,datetime,R_roi,G_roi,B_roi,S_mean,B_R_ratio,contrast,image_path
0,2026-03-01 03:00:00,86.929729,103.464931,122.421467,0.289659,1.408281,19.216036,data\raw\images\20260301-0400.jpg
1,2026-03-01 04:00:00,86.585585,103.941114,123.104967,0.295599,1.421772,14.566550,data\raw\images\20260301-0500.jpg
2,2026-03-01 05:00:00,88.296336,103.318508,121.759842,0.273444,1.378991,13.688991,data\raw\images\20260301-0600.jpg
3,2026-03-01 06:00:00,97.633802,127.312248,157.806914,0.381747,1.616314,17.681540,data\raw\images\20260301-0700.jpg
4,2026-03-01 07:00:00,112.164859,144.062795,172.776857,0.352871,1.540383,21.163309,data\raw\images\20260301-0800.jpg


## Cell 8 — Download PM2.5 data


In [29]:
pm25_df = download_pm25_data(
    api_start=API_START,
    api_end=API_END,
    station_prefix=STATION_PREFIX,
    temp_dir=PM25_TEMP_DIR,
    output_file=PM25_OUTPUT_FILE,
)

pm25_df.head()


Request: {'countries': ['IT'], 'cities': ['Milano (greater city)'], 'pollutants': ['PM2.5'], 'dataset': 1, 'dateTimeStart': '2026-03-01T00:00:00Z', 'dateTimeEnd': '2026-03-12T23:59:59Z', 'aggregationType': 'hour', 'source': 'Jupyter notebook'}
Found parquet files: 6
[1/6] Downloading SPO.IT1650A_6001_BETA_2022-01-01_00_00_00.parquet
[2/6] Downloading SPO.IT0477A_6001_BETA_2022-01-01_00_00_00.parquet
[3/6] Downloading SPO.IT1692A_6001_BETA_2022-01-01_00_00_00.parquet
[4/6] Downloading SPO.IT0480A_6001_BETA_2022-01-01_00_00_00.parquet
[5/6] Downloading SPO.IT1743A_6001_BETA_2022-01-01_00_00_00.parquet
[6/6] Downloading SPO.IT1016A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT0477A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT0480A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT1016A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT1650A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT1692A_6001_BETA_2022-01-01_00_00_00.parquet
Reading SPO.IT1743A_6001_BETA_2022-

,Samplingpoint,Pollutant,Start,End,Value,Unit,AggType,Validity,Verification,ResultTime,DataCapture,FkObservationLog
0,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 00:00:00,2026-03-01 01:00:00,36.062183000000000000,ug.m-3,hour,3,3,2026-03-01 01:00:00,None,9e664d80-8d0a-481e-82ca-112dac15e708
1,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 01:00:00,2026-03-01 02:00:00,25.287216000000000000,ug.m-3,hour,3,3,2026-03-01 02:00:00,None,1499e592-3c02-422b-8646-9dfc91309a4e
2,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 02:00:00,2026-03-01 03:00:00,26.743391000000000000,ug.m-3,hour,3,3,2026-03-01 03:00:00,None,ce0352c6-6450-4676-acd5-03581314f3f2
3,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 03:00:00,2026-03-01 04:00:00,30.829412000000000000,ug.m-3,hour,3,3,2026-03-01 04:00:00,None,d7edb6da-fdba-4913-b409-5fc4b3058c28
4,IT/SPO.IT0477A_6001_BETA_2022-01-01_00:00:00,6001,2026-03-01 04:00:00,2026-03-01 05:00:00,33.409030000000000000,ug.m-3,hour,3,3,2026-03-01 05:00:00,None,c481d8a8-c512-4069-a081-8057f828c6ac


## Cell 9 — Download ERA5 data


In [30]:
era5_df = download_era5_data(
    lat=LAT,
    lon=LON,
    start_date=START_DATE,
    end_date=END_DATE,
    work_dir=ERA5_WORK_DIR,
    output_file=ERA5_OUTPUT_FILE,
    pressure_area=PRESSURE_AREA,
)

era5_df.head()


2026-05-17 17:56:40,020 INFO [2026-05-14T00:00:00Z] Upcoming essential maintenance sessions on Data Stores underlying infrastructure on 19 May. Service disruption expected. For further details, please [visit our forum announcement](https://forum.ecmwf.int/t/upcoming-essential-maintenance-sessions-on-data-stores-underlying-infrastructure/14954).
2026-05-17 17:56:40,212 INFO [2026-02-16T00:00:00] - To generate this ERA5 hourly time series dataset, **homogenisation conventions have been applied to the ERA5 source GRIB data** to ensure consistency, usability, and alignment across chosen variables and time steps. The processed data were then written to an **ARCO Zarr archive**, enabling efficient cloud-optimised access and scalable data retrieval. Please refer to the [user guide](https://confluence.ecmwf.int/x/R6cfHg) for details.

- The dataset presented here is a subset of selected parameters from the full [CDS ERA5 hourly data on single levels (1940–present)](https://cds.climate.copernic

2026-05-17 17:56:40,213 INFO Request ID is 53351f9e-18ec-402a-94dc-fb0e18bcfeb6
2026-05-17 17:56:40,311 INFO status has been updated to accepted
2026-05-17 17:57:02,091 INFO status has been updated to running
2026-05-17 17:57:13,577 INFO status has been updated to successful


4a835efb1c3f108e2b474a64ce68e45d.zip:   0%|          | 0.00/12.8k [00:00<?, ?B/s]

2026-05-17 17:57:14,467 INFO Request ID is 6f362fde-2c08-45b9-b659-a6ecfbdbce89
2026-05-17 17:57:14,539 INFO status has been updated to accepted
2026-05-17 17:57:36,306 INFO status has been updated to successful


5cedeb0463917a12d0afbe87cf657373.zip:   0%|          | 0.00/94.2k [00:00<?, ?B/s]

Single extracted file: D:\eo\GeoProject_oo\data\raw\era5\single_extracted\reanalysis-era5-single-levels-timeseries-sfco6jc2q_w.csv
Pressure extracted file: D:\eo\GeoProject_oo\data\raw\era5\pressure_extracted\data_stream-oper_stepType-instant.nc
Single raw columns: ['valid_time', 'u10', 'v10', 'd2m', 't2m', 'blh', 'cbh', 'sp', 'tcc', 'tp', 'latitude', 'longitude']
Single processed columns: ['time', 'T2M', 'D2M', 'RH', 'U10', 'V10', 'SP', 'TP', 'BLH', 'TCC', 'CBH', 'WS10']
Single processed shape: (288, 12)
Pressure processed columns: ['time', 'GP_500', 'GP_850', 'T_500', 'T_850', 'U_500', 'U_850', 'V_500', 'V_850']
Pressure processed shape: (288, 9)
Saved merged CSV: D:\eo\GeoProject_oo\data\interim\era5_all_merged.csv
Shape: (288, 20)
                  time        T2M        D2M         RH       U10       V10  \
0  2026-03-01 00:00:00  283.19373  281.63806  90.061051 -1.254028 -0.572311   
1  2026-03-01 01:00:00  282.90924  281.53995  91.184942 -0.835602 -0.614883   
2  2026-03-01 02:0

,time,T2M,D2M,RH,U10,V10,SP,TP,BLH,TCC,CBH,WS10,GP_500,GP_850,T_500,T_850,U_500,U_850,V_500,V_850
0,2026-03-01 00:00:00,283.19373,281.63806,90.061051,-1.254028,-0.572311,100798.670,0.000000e+00,31.963226,0.878021,822.22644,1.378451,55447.753906,15084.210938,250.342636,275.176971,9.574036,-0.010620,14.281281,1.972794
1,2026-03-01 01:00:00,282.90924,281.53995,91.184942,-0.835602,-0.614883,100813.670,0.000000e+00,22.768602,0.906525,821.54565,1.037455,55419.699219,15094.523438,250.103119,275.447815,10.276840,0.261047,11.280548,1.783478
2,2026-03-01 02:00:00,282.85132,281.71823,92.651449,-0.774841,0.444016,100812.016,9.536743e-07,28.508335,0.919250,820.26120,0.893045,55359.222656,15084.656250,249.876694,275.475159,9.741043,0.074005,9.810852,1.249084
3,2026-03-01 03:00:00,282.71730,281.67267,93.200513,-0.716339,0.234665,100841.195,1.907349e-06,39.850513,0.813477,396.53946,0.753797,55338.402344,15095.710938,249.700638,275.300751,9.322769,-0.278305,9.539322,0.779373
4,2026-03-01 04:00:00,282.71070,281.80220,94.062501,-1.095779,0.176224,100856.266,5.245209e-06,47.587524,0.965149,252.46167,1.109859,55323.097656,15091.621094,249.479828,275.048279,9.351929,-0.919876,9.568161,0.440033


## Cell 10 — Merge ARPA station tables


In [55]:
arpa_df = merge_arpa_tables(
    files=ARPA_FILES,
    output_file=ARPA_OUTPUT_FILE,
    time_column="Data-Ora",
    sensor_column="Id Sensore",
    utc_offset_hours=1,
    missing_value=-999,
)

arpa_df.head()


Saved: D:\eo\GeoProject_oo\data\interim\arpa_merged.csv
Shape: (3241, 6)
             Data-Ora temperature_mean wind_direction_mean  \
0 2025-12-31 23:00:00              3.3                 248   
1 2026-01-01 00:00:00              2.9                 300   
2 2026-01-01 01:00:00              2.3                 342   
3 2026-01-01 02:00:00              1.9                   4   
4 2026-01-01 03:00:00              1.4                  20   

  relative_humidity_mean wind_speed_mean wind_gust_max  
0                   81.3            <NA>           1.0  
1                   83.0             0.3           1.1  
2                   86.2             0.4           0.9  
3                   87.4            <NA>           1.0  
4                   89.4             0.4           0.9  


,Data-Ora,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean,wind_gust_max
0,2025-12-31 23:00:00,3.3,248,81.3,<NA>,1.0
1,2026-01-01 00:00:00,2.9,300,83.0,0.3,1.1
2,2026-01-01 01:00:00,2.3,342,86.2,0.4,0.9
3,2026-01-01 02:00:00,1.9,4,87.4,<NA>,1.0
4,2026-01-01 03:00:00,1.4,20,89.4,0.4,0.9


## Cell 11 — Merge all datasets
This merges `image_features.csv`, `PM25_MI_hourly.csv`, `era5_all_merged.csv`, and `arpa_merged.csv` by UTC time.


In [59]:
final_df = merge_all_datasets(
    arpa_file=ARPA_OUTPUT_FILE,
    image_file=IMAGE_FEATURE_FILE,
    pm25_file=PM25_OUTPUT_FILE,
    era5_file=ERA5_OUTPUT_FILE,
    output_file=FINAL_OUTPUT_FILE,
    arpa_time_column="Data-Ora",
    pm25_time_column="Start",
    pm25_value_column="Value",
    pm25_unit_column="Unit",
    image_time_column="datetime",
)

final_df.head()


Saved: D:\eo\GeoProject_oo\data\processed\final_merged_all.csv
Final rows: 220
Final shape: (220, 34)
                 time       R_roi       G_roi       B_roi    S_mean  \
0 2026-03-01 03:00:00   86.929729  103.464931  122.421467  0.289659   
1 2026-03-01 04:00:00   86.585585  103.941114  123.104967  0.295599   
2 2026-03-01 05:00:00   88.296336  103.318508  121.759842  0.273444   
3 2026-03-01 06:00:00   97.633802  127.312248  157.806914  0.381747   
4 2026-03-01 07:00:00  112.164859  144.062795  172.776857  0.352871   

   B_R_ratio   contrast                         image_path       PM25    Unit  \
0   1.408281  19.216036  data\raw\images\20260301-0400.jpg  30.829412  ug.m-3   
1   1.421772  14.566550  data\raw\images\20260301-0500.jpg  33.409030  ug.m-3   
2   1.378991  13.688991  data\raw\images\20260301-0600.jpg  45.526115  ug.m-3   
3   1.616314  17.681540  data\raw\images\20260301-0700.jpg  45.983295  ug.m-3   
4   1.540383  21.163309  data\raw\images\20260301-0800.jpg  47.228

,time,R_roi,G_roi,B_roi,S_mean,B_R_ratio,contrast,image_path,PM25,Unit,...,T_850,U_500,U_850,V_500,V_850,temperature_mean,wind_direction_mean,relative_humidity_mean,wind_speed_mean,wind_gust_max
0,2026-03-01 03:00:00,86.929729,103.464931,122.421467,0.289659,1.408281,19.216036,data\raw\images\20260301-0400.jpg,30.829412,ug.m-3,...,275.30075,9.322769,-0.278305,9.539322,0.779373,11.6,73.0,76.2,0.7,2.2
1,2026-03-01 04:00:00,86.585585,103.941114,123.104967,0.295599,1.421772,14.566550,data\raw\images\20260301-0500.jpg,33.409030,ug.m-3,...,275.04828,9.351929,-0.919876,9.568161,0.440033,11.3,84.0,81.4,1.3,3.1
2,2026-03-01 05:00:00,88.296336,103.318508,121.759842,0.273444,1.378991,13.688991,data\raw\images\20260301-0600.jpg,45.526115,ug.m-3,...,274.86044,9.384903,-1.381363,9.571930,0.111649,10.9,74.0,86.8,1.1,2.6
3,2026-03-01 06:00:00,97.633802,127.312248,157.806914,0.381747,1.616314,17.681540,data\raw\images\20260301-0700.jpg,45.983295,ug.m-3,...,274.77585,8.750961,-1.646408,9.686996,-0.082336,10.7,72.0,89.3,1.6,3.3
4,2026-03-01 07:00:00,112.164859,144.062795,172.776857,0.352871,1.540383,21.163309,data\raw\images\20260301-0800.jpg,47.228165,ug.m-3,...,274.84160,8.000458,-1.764099,10.094360,0.228470,10.3,87.0,93.2,2.2,4.2


## Cell 12 — Check final output


In [60]:
print("Final output file:", FINAL_OUTPUT_FILE)
print("Final shape:", final_df.shape)
print("Final columns:")
print(final_df.columns.tolist())


Final output file: D:\eo\GeoProject_oo\data\processed\final_merged_all.csv
Final shape: (220, 34)
Final columns:
['time', 'R_roi', 'G_roi', 'B_roi', 'S_mean', 'B_R_ratio', 'contrast', 'image_path', 'PM25', 'Unit', 'T2M', 'D2M', 'RH', 'U10', 'V10', 'SP', 'TP', 'BLH', 'TCC', 'CBH', 'WS10', 'GP_500', 'GP_850', 'T_500', 'T_850', 'U_500', 'U_850', 'V_500', 'V_850', 'temperature_mean', 'wind_direction_mean', 'relative_humidity_mean', 'wind_speed_mean', 'wind_gust_max']
